# Gold Work Incremental


In [0]:
from pyspark.sql import functions as F 
from delta.tables import DeltaTable
from datetime import datetime 
import uuid 

In [0]:
spark.sql("use catalog  novacart_adb ")

In [0]:
spark.sql("Create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())
run_ts_str= datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print("current date is : ",run_date_str)
print("current time is : ",run_ts_str)
print(gold_run_id)

# spark.sql("use schema gold_schema"

In [0]:
spark.sql("USE CATALOG novacart_adb")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_schema")

spark.sql("""
    CREATE TABLE IF NOT EXISTS 
    novacart_adb.gold_schema.processing_control (
        layer                       STRING,
        entity_name                 STRING,
        last_processed_silver_run_id  STRING,
        last_processed_silver_run_ts  TIMESTAMP,
        rows_merged                 BIGINT,
        run_status                  STRING,
        gold_run_id                 STRING,
        updated_at                  TIMESTAMP
    )
    USING DELTA
""")
print("Gold processing_control created")

In [0]:
def upsert_to_gold(source_df, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
         .merge(
             source_df.alias("source"),
             f"target.{join_key} = source.{join_key}"
         )
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute()
        )
    else:
        source_df.write.format("delta").saveAsTable(target_table)

print("upsert_to_gold defined")

In [0]:
def get_last_processed_silver_ts(entity_name: str):
    ctrl = (
        spark.table("novacart_adb.gold_schema.processing_control")
        .filter(
            (F.col("layer") == "gold") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_silver_run_ts"]

print("get_last_processed_silver_ts defined")

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id,
                         last_processed_silver_ts, rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_silver_ts,
            int(rows_merged),
            "success",
            gold_run_id,
            datetime.utcnow()
        )],
        schema="""
            layer                        STRING,
            entity_name                  STRING,
            last_processed_silver_run_id STRING,
            last_processed_silver_run_ts TIMESTAMP,
            rows_merged                  BIGINT,
            run_status                   STRING,
            gold_run_id                  STRING,
            updated_at                   TIMESTAMP
        """
    )
    dt = DeltaTable.forName(spark, "novacart_adb.gold_schema.processing_control")
    (dt.alias("t")
     .merge(
         ctrl_df.alias("s"),
         "t.layer = s.layer AND t.entity_name = s.entity_name"
     )
     .whenMatchedUpdate(set={
         "last_processed_silver_run_id": "s.last_processed_silver_run_id",
         "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
         "rows_merged":                  "s.rows_merged",
         "run_status":                   "s.run_status",
         "gold_run_id":                  "s.gold_run_id",
         "updated_at":                   "s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute()
    )
    print(f"  Gold control updated for {entity_name}")

print("upsert_gold_control defined")

In [0]:
# Get Gold's last watermark
last_gold_ts = get_last_processed_silver_ts("orders_information")
print("Last processed silver timestamp for gold:", last_gold_ts)

# Read current Silver tables
silver_orders_current   = spark.read.table(
    "novacart_adb.silver_schema.orders_transformed")
silver_products_current = spark.read.table(
    "novacart_adb.silver_schema.products_transformed")
silver_payments_current = spark.read.table(
    "novacart_adb.silver_schema.payments_transformed")

# Filter only changed rows
if last_gold_ts is None:
    # First run — load everything
    changed_orders   = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    # Incremental — only rows newer than last Gold run
    changed_orders   = silver_orders_current.filter(
        F.col("bronze_ingested_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(
        F.col("bronze_ingested_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(
        F.col("bronze_ingested_at") > F.lit(last_gold_ts))

changed_orders_count   = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Changed orders:   {changed_orders_count}")
print(f"Changed products: {changed_products_count}")
print(f"Changed payments: {changed_payments_count}")

In [0]:
# Orders that changed directly
impacted_from_orders = (changed_orders
    .select(F.col("order_id"))
    .distinct())

# Payments that changed — get their order_ids
impacted_from_payments = (changed_payments
    .select(F.col("order_id"))
    .distinct())

# Products that changed — find orders that use those products
impacted_from_products = (
    changed_products.alias("p")
    .join(
        silver_orders_current.alias("o"),
        F.col("p.product_id") == F.col("o.product_id"),
        "inner"
    )
    .select(F.col("o.order_id"))
    .distinct()
)

# Union all impacted order IDs
impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

print(f"Impacted order IDs count: {impacted_order_ids.count()}")
display(impacted_order_ids.orderBy("order_id"))

In [0]:
# Keep only impacted orders from Silver
# Select o.* to drop the duplicate order_id from impacted_order_ids
impacted_orders = (
    silver_orders_current.alias("o")
    .join(
        impacted_order_ids.alias("i"),
        F.col("o.order_id") == F.col("i.order_id"),
        "inner"
    )
    .select("o.*")    # ← THIS IS THE FIX — drops i.order_id duplicate
)

# Join with products and payments
gold_delta = (
    impacted_orders.alias("o")

    .join(silver_products_current.alias("p"),
          F.col("o.product_id") == F.col("p.product_id"),
          "inner")

    .join(silver_payments_current.alias("py"),
          F.col("o.order_id") == F.col("py.order_id"),
          "inner")

    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("py.payment_id"),
        F.col("py.payment_status"),
        F.col("py.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at"),
            F.col("p.updated_at"),
            F.col("py.processed_at")
        ).alias("gold_updated_ts")
    )

    .dropDuplicates(["order_id"])

    .withColumn("payment_completion_ratio",
        F.when(F.col("order_amount") > 0,
            F.col("paid_amount") / F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )

    .withColumn("payment_state",
        F.when(F.col("order_amount") == 0,  "invalid_order_amount")
        .when(F.col("payment_completion_ratio") == 0, "unpaid")
        .when(F.col("payment_completion_ratio") == 1, "paid")
        .when(F.col("payment_completion_ratio") < 1,  "partially_paid")
        .otherwise("overpaid")
    )

    .withColumn("gold_updated_date", F.to_date("gold_updated_ts"))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)

gold_delta_count = gold_delta.count()
print(f"Gold delta rows: {gold_delta_count}")
display(gold_delta)

In [0]:
if gold_delta_count > 0:
    upsert_to_gold(
        gold_delta,
        "novacart_adb.gold_schema.orders_information",
        "order_id"
    )
    print("orders_information updated")
else:
    print("No new records to insert in gold table")

In [0]:
scd2_table = "novacart_adb.gold_schema.orders_information_scd2"

# Create SCD2 table on first run
if not spark.catalog.tableExists(scd2_table):
    spark.sql("""
        CREATE TABLE novacart_adb.gold_schema.orders_information_scd2
        USING DELTA
        AS SELECT *,
            CAST(NULL AS TIMESTAMP) AS valid_from_ts,
            CAST(NULL AS TIMESTAMP) AS valid_to_ts,
            TRUE AS is_current
        FROM novacart_adb.gold_schema.orders_information
        WHERE 1 = 0
    """)
    print("SCD2 table created")

if gold_delta_count > 0:

    gold_delta.createOrReplaceTempView("gold_delta_view")

    spark.sql("""
        MERGE INTO novacart_adb.gold_schema.orders_information_scd2 AS t
        USING gold_delta_view AS s
        ON t.order_id = s.order_id AND t.is_current = TRUE
        WHEN MATCHED AND (
            NOT (t.order_status   <=> s.order_status)   OR
            NOT (t.order_amount   <=> s.order_amount)   OR
            NOT (t.paid_amount    <=> s.paid_amount)    OR
            NOT (t.payment_id     <=> s.payment_id)     OR
            NOT (t.category       <=> s.category)       OR
            NOT (t.product_name   <=> s.product_name)   OR
            NOT (t.product_price  <=> s.product_price)
        )
        THEN UPDATE SET
            is_current   = FALSE,
            valid_to_ts  = s.gold_updated_ts
    """)

    spark.sql("""
        INSERT INTO novacart_adb.gold_schema.orders_information_scd2
        SELECT s.*,
            s.gold_updated_ts AS valid_from_ts,
            CAST(NULL AS TIMESTAMP) AS valid_to_ts,
            TRUE AS is_current
        FROM gold_delta_view s
        LEFT JOIN novacart_adb.gold_schema.orders_information_scd2 t
            ON s.order_id = t.order_id AND t.is_current = TRUE
        WHERE t.order_id IS NULL OR
            NOT (t.order_status  <=> s.order_status)  OR
            NOT (t.order_amount  <=> s.order_amount)  OR
            NOT (t.paid_amount   <=> s.paid_amount)   OR
            NOT (t.payment_id    <=> s.payment_id)    OR
            NOT (t.category      <=> s.category)      OR
            NOT (t.product_name  <=> s.product_name)  OR
            NOT (t.product_price <=> s.product_price)
    """)

    print("SCD2 updated")
# Error explanation and fix: 
# - Replaced {scd2_table} and {gold_delta_view} with explicit table/view names in SQL strings.
# - Removed curly braces in SQL, as Databricks SQL does not support Python string interpolation in triple-quoted SQL.

# Update Category-- > level gold aggregation


In [0]:
if gold_delta_count > 0:

    impacted_categories = (
        gold_delta
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read.table("novacart_adb.gold_schema.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Gross_Merchandise_Value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Total_Paid_Amount"),
            F.avg("payment_completion_ratio").alias(
                "Average_Payment_Completion_Ratio"),
            (
                F.sum(
                    F.when(F.col("payment_status") == "SUCCESS", 1)
                    .otherwise(0)
                ) / F.count("*")
            ).alias("Payment_Failure_Rate")
        )
    )

    upsert_to_gold(
        category_perf_delta,
        "novacart_adb.gold_schema.category_performance",
        "category"
    )
    print("category_performance updated")
    display(spark.table("novacart_adb.gold_schema.category_performance"))

# Publish Gold Snapshots  to Volume 
latest Snapshot -->  overwritten every successful run 
timestamped historical snapshot --> a new  folder for each successful run

In [0]:
spark.sql("create volume if not exists novacart_adb.gold_schema.gold_snapshots_vol")

In [0]:
# Define paths
vol = "/Volumes/novacart_adb/gold_schema/gold_snapshots_vol"

latest_orders_path   = f"{vol}/gold_latest/orders_information"
latest_category_path = f"{vol}/gold_latest/category_performance"

historical_orders_path   = f"{vol}/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"
historical_category_path = f"{vol}/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"

# Write latest snapshots
(spark.read.table("novacart_adb.gold_schema.orders_information")
 .write.format("parquet").mode("overwrite")
 .save(latest_orders_path))

(spark.read.table("novacart_adb.gold_schema.category_performance")
 .write.format("parquet").mode("overwrite")
 .save(latest_category_path))

# Write historical snapshots
(spark.read.table("novacart_adb.gold_schema.orders_information")
 .write.format("parquet").mode("overwrite")
 .save(historical_orders_path))

(spark.read.table("novacart_adb.gold_schema.category_performance")
 .write.format("parquet").mode("overwrite")
 .save(historical_category_path))

print("Latest orders path    :", latest_orders_path)
print("Latest category path  :", latest_category_path)
print("Historical orders path:", historical_orders_path)
print("Historical category   :", historical_category_path)

In [0]:
latest_silver_ts = (
    silver_orders_current
    .agg(F.max("bronze_ingested_at").alias("mx"))
    .collect()[0]["mx"]
)

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == F.lit(latest_silver_ts))
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta.count()
)

display(spark.table("novacart_adb.gold_schema.processing_control"))